# Mã nguồn cào dữ liệu từ ShopeeFood

## **I. Import Libraries Necessary**

In [1]:
import re
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
import random
from selenium.webdriver.common.action_chains import ActionChains
from datetime import datetime
import pickle
import time
from shopee_tools import *

In [2]:
chrome_options = Options()

# Ẩn flag "Chrome is being controlled by automated test software"
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

### Vui lòng điền link địa điểm cần cào qua tham số **url**

In [3]:
driver = webdriver.Chrome(options = chrome_options)
driver.maximize_window()
url = 'https://shopeefood.vn/dong-nai/danh-sach-dia-diem-giao-tan-noi?q=tr%C3%A0%20s%E1%BB%AFa' # Có thể thay đổi link tùy theo khu vực
driver.get(url)
human_action_simulation(driver)
time.sleep(3) # Đợi hiện số Kết quả ở góc trên bên phải

* Số quán thu thập được

In [4]:
count_item = driver.find_element(By.XPATH, '//*[@id="app"]/div/div[1]/div[1]/div[1]/div[2]/div')
count_item = count_item.text.split(' ')[0]
count_item

'5'

In [5]:
i = 1
list_links = []

# Extract all elements (restaurants) from the first page. //*[@id="app"]/div/div[1]/div[3]/div/div[2]
div_list_restaurant = driver.find_elements(By.XPATH, '//div[@class="now-list-restaurant"]//div[@class="list-restaurant"]/div[@class="item-restaurant"]')
# Implement a loop to iterate through each element item within the div_list_restaurant class on the first page.
for div_restaurant in div_list_restaurant:
    try:
        link = div_restaurant.find_element(By.XPATH, 'a[@class="item-content"]').get_attribute('href')
        # Append the extracted link variable to the list_link to create a list of links for accessing each food item.
        list_links.append(link + '\n')
    except:
        list_links.append('link error' + '\n')
print('Page', i)
i += 1 
# Employ a while loop to iterate through all remaining pages.
while True:
    try:
        # Click the "Next" button to navigate to the next page and wait for 10 seconds.
        next_button = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "span.icon.icon-paging-next"))
    )
        driver.execute_script("arguments[0].click();", next_button)
        time.sleep(4)
        print('Page', i)
        i += 1
        
        # Upon loading each new page, repeat the same actions as performed on the first page.
        div_list_restaurant = driver.find_elements(By.XPATH, '//div[@class="now-list-restaurant"]//div[@class="list-restaurant"]/div[@class="item-restaurant"]')
        # Implement a loop to iterate through each element item within the div_list_restaurant class.
        for div_restaurant in div_list_restaurant:
            # For each item in the PATH, utilize the find_element method to retrieve the hyperlink from the a tag. Extract the link from the href attribute and assign it to the link variable.
            try:
                link = div_restaurant.find_element(By.XPATH, 'a[@class="item-content"]').get_attribute('href')
                # Append the extracted link variable to the list_link to create a list of links for accessing each food item.
                list_links.append(link + '\n')
            except:
                list_links.append('link error' + '\n')
        
        # If the list of restaurant links matches the total number of restaurants on the website, terminate the process, save the link list to a txt file, and close the browser.
        if len(list_links) == int(count_item):
            write_file_txt('../../data/data_raw/txt/link_food_hcm.txt', list_links)
            driver.close()
            print('Done')
            break 
        else:
            continue
    except Exception as e:
        write_file_txt('../../data/data_raw/txt/link_food_hcm.txt', list_links)
        driver.close()
        print('Done with error (No next button) !')
        break

Page 1
Done with error (No next button) !


In [6]:
len(list_links)

5

#### 2. Utilize the recently created function to retrieve the data.

In [7]:
list_links = read_file_txt('../../data/data_raw/txt/link_food_hcm.txt')
restaurant_df, review_df = get_data(list_links, 1, 1, chrome_options) # Lấy hết link trong file data_raw/txt/link_food_hcm.txt

current: 1
current: 2
current: 3
current: 4
current: 5


### Vui lòng đặt số hiệu của file đầu ra

In [ ]:
# Số hiệu
outname = 1000
# Thông tin cửa hàng: D
# Đặt tên theo định dạng D{số thứ tự}.csv như bên dưới
restaurant_df.to_csv(f'../../data/data_raw/shopee_csv/D/D{outname}.csv', index=False, encoding='utf-8-sig')

# Bình luận: C
# Đặt tên theo định dạng C{số thứ tự}.csv như bên dưới
review_df.to_csv(f'../../data/data_raw/shopee_csv/C/C{outname}.csv', index=False, encoding='utf-8-sig')